# Multimodal Product Categorization

Classify fashion products into 20 categories from their **image and title**, and measure how much each modality contributes. The model fuses a DistilBERT text encoder with a ResNet-50 image encoder.

Product titles usually name the category outright, so text alone scores high. The more interesting question is robustness: if the title is missing, can the model still classify from the image? Each arm — text-only, image-only, and fusion — is scored with the title present and removed.

**Setup:** turn on GPU and Internet, and attach the *Fashion Product Images (Small)* dataset.

In [31]:
# Kaggle ships torch / torchvision / scikit-learn; just refresh transformers.
!pip -q install -U transformers
import torch, transformers
print('transformers', transformers.__version__, '| torch', torch.__version__,
      '| GPU available:', torch.cuda.is_available())

transformers 5.12.0 | torch 2.10.0+cu128 | GPU available: True


## Configuration

Hyperparameters and paths.

In [32]:
"""Central configuration for the multimodal product categorization project.

Every tunable lives here so the notebook, the CLI runner, and the Gradio app
all read the same knobs. The defaults are tuned for a *fast first run* on a
Kaggle GPU (frozen backbones, subsampled data); flip the marked flags for the
full training run that produces the full results.
"""
import os
import torch

# ---- Reproducibility ----
SEED = 42

# ---- Data: Fashion Product Images (Kaggle) ----
# One click on Kaggle: Add Data -> "Fashion Product Images (Small)"
# (paramaggarwal/fashion-product-images-small). The loader recursively finds
# styles.csv + the images/ folder, so the exact mount nesting doesn't matter.
DATA_ROOT = os.environ.get("DATA_ROOT", "/kaggle/input/fashion-product-images-small")
LABEL_COL = "articleType"    # what to predict. "articleType" (many, harder, best
                             # fusion story), "subCategory" (~45), "masterCategory" (~7, easy).
TOP_K_CATEGORIES = 20        # keep the K most frequent classes
MAX_PER_CLASS = 1500         # subsample each class -> fast first run
VAL_FRAC = 0.15
TEST_FRAC = 0.15

# If the dataset isn't found, generate a tiny synthetic set so the WHOLE pipeline still
# runs end-to-end (the numbers are meaningless). This lets
# you checking the code path before attaching the real dataset.
USE_SYNTHETIC_FALLBACK = True
SYNTHETIC_N = 1200

# ---- Text encoder ----
TEXT_MODEL = "distilbert-base-uncased"
MAX_TEXT_LEN = 48

# ---- Image encoder ----
IMAGE_SIZE = 224
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

# ---- Training ----
# Defaults do a real FINE-TUNING run (unfrozen encoders) for strong results.
# For a quick check instead, set FREEZE_BACKBONE = True (trains only the head; weak numbers).
BATCH_SIZE = 32          # fits two fine-tuned backbones on a 16 GB T4; drop to 16 if you hit OOM
EPOCHS = 5
FREEZE_BACKBONE = False
HEAD_LR = 1e-3           # classifier head learns fast
BACKBONE_LR = 2e-5       # encoders fine-tune gently (unused when FREEZE_BACKBONE=True)
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2

# Robustness experiment: product titles often spell out the category, so text alone
# nearly maxes out. To show the IMAGE earns its keep, we train the fusion model with
# random "missing title" augmentation, then test every arm with titles present vs.
# removed. Fusion should stay strong without titles (it falls back on the image),
# while text-only collapses.
TEXT_DROPOUT = 0.5       # fraction of titles blanked during fusion training

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ARTIFACT_DIR = os.environ.get("ARTIFACT_DIR", "artifacts")

print("Config ready  |  device:", DEVICE, "  |  dataset:", DATA_ROOT)

Config ready  |  device: cuda   |  dataset: /kaggle/input/fashion-product-images-small


## Data

Read the product metadata and images and keep the most common categories. If the dataset isn't attached, a small synthetic set lets the notebook still run end to end.

In [33]:
"""Fashion Product Images (Kaggle) parsing + the PyTorch dataset.

The dataset ships a `styles.csv` (product metadata incl. productDisplayName and
category columns) plus an `images/<id>.jpg` folder. We join them into clean
(title, category, image_path) rows, keep the top-K categories, and subsample.

If the dataset isn't present we fall back to a small synthetic set so the rest
of the pipeline still runs (see config.USE_SYNTHETIC_FALLBACK).
"""
import os
import glob
import random

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset
import torchvision.transforms as T

def _find_file(root, name):
    hits = glob.glob(os.path.join(root, "**", name), recursive=True)
    return hits[0] if hits else None

def _find_images_dir(root):
    for h in glob.glob(os.path.join(root, "**", "images"), recursive=True):
        if os.path.isdir(h):
            return h
    return None

def load_fashion(data_root):
    """Return DataFrame[title, category, image_path] from the Fashion Product Images dataset."""
    styles = _find_file(data_root, "styles.csv")
    images_dir = _find_images_dir(data_root)
    if not styles or not images_dir:
        return pd.DataFrame(columns=["title", "category", "image_path"])
    # productDisplayName (the last column) can contain unescaped commas, so a naive
    # CSV read drops those rows. Split with a capped maxsplit so the final field
    # absorbs any extra commas -> we keep every row instead of skipping them.
    with open(styles, encoding="utf-8") as f:
        header = [c.strip() for c in f.readline().rstrip("\n").split(",")]
        ncol = len(header)
        recs = [parts for line in f
                if len(parts := line.rstrip("\n").split(",", ncol - 1)) == ncol]
    df = pd.DataFrame(recs, columns=header)
    if LABEL_COL not in df.columns or "productDisplayName" not in df.columns:
        return pd.DataFrame(columns=["title", "category", "image_path"])
    df = df[["id", "productDisplayName", LABEL_COL]]
    df = df[(df["productDisplayName"].str.len() > 0) & (df[LABEL_COL].str.len() > 0)]
    df["image_path"] = df["id"].map(lambda i: os.path.join(images_dir, f"{i}.jpg"))
    df = df[df.image_path.map(os.path.exists)]
    df = df.rename(columns={"productDisplayName": "title", LABEL_COL: "category"})
    return df[["title", "category", "image_path"]]

def _synthetic_dataframe():
    """Tiny fake dataset: titles literally contain the category, images are noise.
    Text-only will score well, image-only ~chance, fusion ~text. quick check only."""
    cats = [f"CATEGORY_{i}" for i in range(min(TOP_K_CATEGORIES, 8))]
    words = ["cotton", "steel", "wooden", "wireless", "leather", "ceramic", "plastic",
             "cordless", "vintage", "compact", "premium", "portable", "stainless", "ergonomic"]
    rng = random.Random(SEED)
    rows = []
    for i in range(SYNTHETIC_N):
        c = rng.choice(cats)
        title = " ".join(rng.sample(words, 4)) + f" {c.lower()}"
        rows.append((f"syn_{i}", title, c, None))
    return pd.DataFrame(rows, columns=["image_id", "title", "category", "image_path"])

def build_dataframe():
    """Load + clean the dataset, keep top-K categories, subsample. Returns (df, is_synthetic)."""
    df = pd.DataFrame()
    for root in [DATA_ROOT, "/kaggle/input"]:   # try the configured path, then any attached dataset
        if root and os.path.exists(root):
            df = load_fashion(root)
            if len(df):
                print(f"[data] Loaded real dataset from: {root}")
                break

    if len(df) == 0:
        if not USE_SYNTHETIC_FALLBACK:
            raise RuntimeError(
                f"No dataset found under {DATA_ROOT}. Attach 'Fashion Product Images (Small)' "
                f"or set USE_SYNTHETIC_FALLBACK=True."
            )
        print("=" * 72)
        print("  WARNING: Fashion dataset NOT FOUND  ->  using SYNTHETIC dummy data.")
        print("  The numbers below are MEANINGLESS (text=100% by construction, image=chance).")
        print("  Fix on Kaggle:  Add Data (right panel) -> search")
        print("  'Fashion Product Images Small' (by paramaggarwal) -> Add -> Run All again.")
        print("=" * 72)
        return _synthetic_dataframe(), True

    top = df.category.value_counts().head(TOP_K_CATEGORIES).index
    df = df[df.category.isin(top)]
    # Subsample each class via explicit concat (robust across pandas versions;
    # groupby.apply can fold the grouping column into the index).
    parts = [g.sample(min(len(g), MAX_PER_CLASS), random_state=SEED)
             for _, g in df.groupby("category")]
    df = pd.concat(parts).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    return df, False

def make_splits(df):
    """Shuffle and carve into train/val/test. Returns (train, val, test, label2idx, idx2label)."""
    labels = sorted(df.category.unique())
    label2idx = {c: i for i, c in enumerate(labels)}
    idx2label = {i: c for c, i in label2idx.items()}
    df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    n = len(df)
    n_test = int(n * TEST_FRAC)
    n_val = int(n * VAL_FRAC)
    test = df.iloc[:n_test]
    val = df.iloc[n_test:n_test + n_val]
    train = df.iloc[n_test + n_val:]
    return train, val, test, label2idx, idx2label

def make_transforms(train):
    if train:
        return T.Compose([
            T.RandomResizedCrop(IMAGE_SIZE, scale=(0.7, 1.0)),
            T.RandomHorizontalFlip(),
            T.ToTensor(),
            T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    return T.Compose([
        T.Resize(int(IMAGE_SIZE * 1.14)),
        T.CenterCrop(IMAGE_SIZE),
        T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

class ProductDataset(Dataset):
    """Yields {input_ids, attention_mask, image, label} for one product."""

    def __init__(self, df, tokenizer, label2idx, train=False, text_dropout=0.0, blank_text=False):
        self.df = df.reset_index(drop=True)
        self.tok = tokenizer
        self.label2idx = label2idx
        self.tf = make_transforms(train)
        self.text_dropout = text_dropout   # train-time: randomly blank this fraction of titles
        self.blank_text = blank_text        # eval-time: blank every title (simulate missing metadata)

    def __len__(self):
        return len(self.df)

    def _load_image(self, path):
        try:
            if path and os.path.exists(path):
                return Image.open(path).convert("RGB")
        except Exception:
            pass
        # Synthetic / missing image -> random noise (keeps the transform path uniform).
        arr = (np.random.rand(IMAGE_SIZE, IMAGE_SIZE, 3) * 255).astype("uint8")
        return Image.fromarray(arr)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        image = self.tf(self._load_image(r.get("image_path", None)))
        title = str(r["title"])
        # Blank the title when simulating missing metadata (eval) or for training-time
        # dropout. An empty string still tokenizes to [CLS][SEP] -> a valid "no text" input.
        if self.blank_text or (self.text_dropout > 0 and random.random() < self.text_dropout):
            title = ""
        enc = self.tok(
            title, truncation=True, max_length=MAX_TEXT_LEN,
            padding="max_length", return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "image": image,
            "label": torch.tensor(self.label2idx[r["category"]], dtype=torch.long),
        }

print("Data layer ready: build_dataframe(), make_splits(), ProductDataset")

Data layer ready: build_dataframe(), make_splits(), ProductDataset


## Models

One model class covers all three arms — text-only, image-only, and fusion — so the comparison is apples-to-apples: same head, same training, only the inputs change.

In [34]:
"""Text, image, and fusion encoders.

One `MultiModalModel` class serves all three ablation arms (text-only,
image-only, both) so the comparison is apples-to-apples: same head, same
training, only the inputs differ.
"""
import torch
import torch.nn as nn
import torchvision
from transformers import AutoModel

class TextEncoder(nn.Module):
    """DistilBERT -> 768-d [CLS] embedding."""
    out_dim = 768

    def __init__(self, freeze=True):
        super().__init__()
        self.bert = AutoModel.from_pretrained(TEXT_MODEL)
        if freeze:
            for p in self.bert.parameters():
                p.requires_grad = False

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        return out.last_hidden_state[:, 0]  # DistilBERT has no pooler; use [CLS] position

class ImageEncoder(nn.Module):
    """ResNet-50 (ImageNet) -> 2048-d embedding."""
    out_dim = 2048

    def __init__(self, freeze=True):
        super().__init__()
        m = torchvision.models.resnet50(
            weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2
        )
        m.fc = nn.Identity()
        self.backbone = m
        if freeze:
            for p in self.backbone.parameters():
                p.requires_grad = False

    def forward(self, image):
        return self.backbone(image)

class MultiModalModel(nn.Module):
    """modality in {'text', 'image', 'both'}. Builds only the encoders it needs."""

    def __init__(self, modality, num_classes, freeze=True, hidden=512, dropout=0.3):
        super().__init__()
        assert modality in {"text", "image", "both"}
        self.modality = modality
        in_dim = 0
        if modality in {"text", "both"}:
            self.text = TextEncoder(freeze)
            in_dim += TextEncoder.out_dim
        if modality in {"image", "both"}:
            self.image = ImageEncoder(freeze)
            in_dim += ImageEncoder.out_dim
        self.head = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Dropout(dropout),
            nn.Linear(in_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_classes),
        )

    def forward(self, batch):
        feats = []
        if self.modality in {"text", "both"}:
            feats.append(self.text(batch["input_ids"], batch["attention_mask"]))
        if self.modality in {"image", "both"}:
            feats.append(self.image(batch["image"]))
        x = torch.cat(feats, dim=1) if len(feats) > 1 else feats[0]
        return self.head(x)

print("Models ready: MultiModalModel(modality='text'|'image'|'both')")

Models ready: MultiModalModel(modality='text'|'image'|'both')


## Training and evaluation

In [35]:
"""Train / evaluate loops and the ablation runner.

`run_ablation` trains the three arms in turn and returns a tidy results table —
that table (fusion beating both single-modality baselines) is the headline.
"""
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score

def _to_device(batch):
    return {k: v.to(DEVICE) for k, v in batch.items()}

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    ys, ps = [], []
    for batch in loader:
        batch = _to_device(batch)
        logits = model(batch)
        ps.append(logits.argmax(1).cpu().numpy())
        ys.append(batch["label"].cpu().numpy())
    y = np.concatenate(ys)
    p = np.concatenate(ps)
    return accuracy_score(y, p), f1_score(y, p, average="macro")

def train_model(modality, train_ds, val_ds, num_classes):
    model = MultiModalModel(modality, num_classes, freeze=FREEZE_BACKBONE).to(DEVICE)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
    # Discriminative learning rates: the fresh classifier head learns fast, while the
    # pretrained encoders fine-tune gently so we don't wash out their ImageNet/BERT priors.
    head_params, backbone_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (head_params if name.startswith("head") else backbone_params).append(p)
    groups = [{"params": head_params, "lr": HEAD_LR}]
    if backbone_params:
        groups.append({"params": backbone_params, "lr": BACKBONE_LR})
    opt = torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)
    crit = nn.CrossEntropyLoss()

    best_f1, best_state = -1.0, None
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running = 0.0
        for batch in train_loader:
            batch = _to_device(batch)
            opt.zero_grad()
            loss = crit(model(batch), batch["label"])
            loss.backward()
            opt.step()
            running += loss.item()
        acc, f1 = evaluate(model, val_loader)
        print(f"[{modality:5}] epoch {epoch}/{EPOCHS}  "
              f"loss {running / max(len(train_loader), 1):.3f}  "
              f"val_acc {acc:.3f}  val_f1 {f1:.3f}")
        if f1 > best_f1:
            best_f1 = f1
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
    if best_state:
        model.load_state_dict(best_state)
    return model

def run_ablation(train_df, val_df, test_df, tokenizer, label2idx):
    num_classes = len(label2idx)
    val_ds = ProductDataset(val_df, tokenizer, label2idx, train=False)
    test_full = ProductDataset(test_df, tokenizer, label2idx, train=False)
    test_blank = ProductDataset(test_df, tokenizer, label2idx, train=False, blank_text=True)

    def _acc(model, ds):
        loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
        return evaluate(model, loader)[0]

    results, models = [], {}
    for modality in ["text", "image", "both"]:
        print(f"\n=== Training arm: {modality} ===")
        # Fusion trains with text dropout so it learns to fall back on the image when the
        # title is missing; the single-modality baselines train on clean data.
        td = TEXT_DROPOUT if modality == "both" else 0.0
        train_ds = ProductDataset(train_df, tokenizer, label2idx, train=True, text_dropout=td)
        model = train_model(modality, train_ds, val_ds, num_classes)
        acc_full = _acc(model, test_full)
        # Blanking the title is meaningless for the image-only model -> reuse its score.
        acc_blank = acc_full if modality == "image" else _acc(model, test_blank)
        results.append({"modality": modality,
                        "acc_full_title": round(acc_full, 4),
                        "acc_no_title": round(acc_blank, 4)})
        models[modality] = model
        print(f"--> {modality}: full-title {acc_full:.3f}  |  no-title {acc_blank:.3f}")

    res = pd.DataFrame(results)
    # Headline: when titles vanish, text collapses but fusion holds (it uses the image).
    t = res.set_index("modality")
    gain = (t.loc["both", "acc_no_title"] - t.loc["text", "acc_no_title"]) * 100
    print(f"\nWith titles:    text {t.loc['text','acc_full_title']:.3f}   fusion {t.loc['both','acc_full_title']:.3f}")
    print(f"Titles removed: text {t.loc['text','acc_no_title']:.3f}   fusion {t.loc['both','acc_no_title']:.3f}"
          f"   -> fusion is +{gain:.0f} pts more robust")
    return res, models

print("Training engine ready: run_ablation()")

Training engine ready: run_ablation()


## Train and evaluate

Train all three arms, then score each with the title present and removed.

In [36]:
import random
import numpy as np
import torch
from transformers import AutoTokenizer

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print("Device:", DEVICE)

df, is_synth = build_dataframe()
print(f"Loaded {len(df)} products / {df.category.nunique()} categories (synthetic={is_synth})")
display(df.head())
print(df.category.value_counts().head(10))

train_df, val_df, test_df, label2idx, idx2label = make_splits(df)
print(f"train/val/test = {len(train_df)}/{len(val_df)}/{len(test_df)}  |  {len(label2idx)} classes")

tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL)
results_df, models = run_ablation(train_df, val_df, test_df, tokenizer, label2idx)
results_df

Device: cuda
[data] Loaded real dataset from: /kaggle/input
Loaded 22076 products / 20 categories (synthetic=False)


,title,category,image_path
0,Flying Machine Women Solid Pink Tops,Tops,/kaggle/input/datasets/paramaggarwal/fashion-p...
1,U.S. Polo Assn. Men Checks Maroon Shirt,Shirts,/kaggle/input/datasets/paramaggarwal/fashion-p...
2,Rockport Men Tr Bal Brown Casual Shoes,Casual Shoes,/kaggle/input/datasets/paramaggarwal/fashion-p...
3,U.S. Polo Assn. Men White Flip Flops,Flip Flops,/kaggle/input/datasets/paramaggarwal/fashion-p...
4,Arrow Woman Beige Top,Tops,/kaggle/input/datasets/paramaggarwal/fashion-p...


category
Tops            1500
Shirts          1500
Casual Shoes    1500
Handbags        1500
Watches         1500
Sports Shoes    1500
Kurtas          1500
Tshirts         1500
Heels           1323
Sunglasses      1073
Name: count, dtype: int64
train/val/test = 15454/3311/3311  |  20 classes

=== Training arm: text ===


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[text ] epoch 1/5  loss 0.218  val_acc 0.975  val_f1 0.976
[text ] epoch 2/5  loss 0.073  val_acc 0.980  val_f1 0.981
[text ] epoch 3/5  loss 0.059  val_acc 0.975  val_f1 0.978
[text ] epoch 4/5  loss 0.049  val_acc 0.977  val_f1 0.979
[text ] epoch 5/5  loss 0.041  val_acc 0.976  val_f1 0.978
--> text: full-title 0.975  |  no-title 0.028

=== Training arm: image ===
[image] epoch 1/5  loss 0.539  val_acc 0.895  val_f1 0.905
[image] epoch 2/5  loss 0.300  val_acc 0.906  val_f1 0.915
[image] epoch 3/5  loss 0.241  val_acc 0.905  val_f1 0.915
[image] epoch 4/5  loss 0.201  val_acc 0.917  val_f1 0.925
[image] epoch 5/5  loss 0.179  val_acc 0.919  val_f1 0.927
--> image: full-title 0.905  |  no-title 0.905

=== Training arm: both ===


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[both ] epoch 1/5  loss 0.429  val_acc 0.978  val_f1 0.980
[both ] epoch 2/5  loss 0.221  val_acc 0.979  val_f1 0.981
[both ] epoch 3/5  loss 0.190  val_acc 0.982  val_f1 0.984
[both ] epoch 4/5  loss 0.159  val_acc 0.976  val_f1 0.979
[both ] epoch 5/5  loss 0.150  val_acc 0.982  val_f1 0.984
--> both: full-title 0.980  |  no-title 0.886

With titles:    text 0.975   fusion 0.980
Titles removed: text 0.028   fusion 0.886   -> fusion is +86 pts more robust


,modality,acc_full_title,acc_no_title
0,text,0.9749,0.0278
1,image,0.9052,0.9052
2,both,0.9801,0.8861


## Save the model

In [39]:
import os
import json

os.makedirs(ARTIFACT_DIR, exist_ok=True)
torch.save(models["both"].state_dict(), os.path.join(ARTIFACT_DIR, "fusion_model.pt"))
with open(os.path.join(ARTIFACT_DIR, "labels.json"), "w") as f:
    json.dump(idx2label, f)
results_df.to_csv(os.path.join(ARTIFACT_DIR, "ablation_results.csv"), index=False)
print("Saved ->", ARTIFACT_DIR, os.listdir(ARTIFACT_DIR))

Saved -> artifacts ['ablation_results.csv', 'labels.json', 'fusion_model.pt']


## Ideas to extend

- Fine-tune the encoders (`FREEZE_BACKBONE = False`) for higher accuracy.
- Use more data (raise `MAX_PER_CLASS`, add categories).
- Add a cosine learning-rate schedule.
- Wrap the fusion model in a small demo.